# CoinSprout: Step 3, Allocation Engine
Takes the **risk category** (Step 1) plus **mu and cov** (Step 2) and outputs a percentage split across MMF and Equities.

**What is "trained" here:** nothing. This is mean-variance optimisation (a solver), not machine learning. Only Step 1 is trained. Say this in your defence.

**How it works**
1. Maximise `mu·w − (λ/2)·wᵀΣw`. λ is risk aversion: higher λ means a more cautious split.
2. Each category has weight bounds. Conservative's equity cap is *below* Aggressive's equity floor, so Conservative can never be riskier than Aggressive by construction.
3. A stability test re-runs the allocator on the 200 bootstrap versions of mu and cov from Step 2.

**Educational use only. Not financial advice.**

## 1. Load Step 2 output

In [ ]:
import json, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

STATS_PATH = "asset_stats.json"   # upload the file Step 2 saved (or put it next to this notebook)
USING_STANDIN = False
if not os.path.exists(STATS_PATH):
    try:
        from google.colab import files
        print("Upload asset_stats.json from Step 2 (cancel to use stand-in numbers)")
        up = files.upload()
        if "asset_stats.json" in up: STATS_PATH = "asset_stats.json"
    except Exception:
        pass

if os.path.exists(STATS_PATH):
    S = json.load(open(STATS_PATH))
    ASSETS = S["assets"]
    mu = np.array(S["mu"]); cov = np.array(S["cov"])
    boot_mu = np.array(S["bootstrap_mu"]); boot_cov = np.array(S["bootstrap_cov"])
else:
    USING_STANDIN = True
    print("*** STAND-IN NUMBERS, NOT REAL DATA. Results below are for testing the code only. ***")
    rng = np.random.default_rng(0)
    ASSETS = ["MMF", "Equity"]
    mu = np.array([0.10, 0.09])
    sd = np.array([0.01, 0.18]); corr = np.array([[1,.1],[.1,1]])
    cov = np.outer(sd, sd) * corr
    L = np.linalg.cholesky(cov)
    boot_mu, boot_cov = [], []
    for _ in range(200):
        x = rng.standard_normal((120, 2)) @ L.T / np.sqrt(12) + mu / 12
        boot_mu.append(x.mean(0) * 12); boot_cov.append(np.cov(x.T) * 12)
    boot_mu, boot_cov = np.array(boot_mu), np.array(boot_cov)

# find the equity column by name so the order in the JSON does not matter
EQ = next(i for i, a in enumerate(ASSETS) if any(k in a.lower() for k in ("equit", "nasi", "nse", "stock")))
print("Assets:", ASSETS, "| equity index:", EQ)
print("mu (%):", (mu * 100).round(2))
print("volatility (%):", (np.sqrt(np.diag(cov)) * 100).round(2))

## 2. Risk categories: settings you can defend
Edit these and say why in the report. They are **policy choices**, not learned values.

| Category | λ (risk aversion) | Equity bounds | Other bounds |
|---|---|---|---|
| Conservative | 12 | 0 to 15% | none (two assets, so MMF = 100% minus equity) |
| Moderate | 5 | 10 to 35% | none |
| Aggressive | 1.5 | 25 to 60% | none |

Because Conservative's cap (15%) is below Aggressive's floor (25%), the ordering is guaranteed.

In [ ]:
CONFIG = {
    "Conservative": {"lam": 12.0, "eq": (0.00, 0.15), "max_w": 1.0},
    "Moderate":     {"lam": 5.0,  "eq": (0.10, 0.35), "max_w": 1.0},
    "Aggressive":   {"lam": 1.5,  "eq": (0.25, 0.60), "max_w": 1.0},
}
MIN_W = 0.0   # raise to e.g. 0.05 to force every asset to appear in every split
assert CONFIG["Conservative"]["eq"][1] < CONFIG["Aggressive"]["eq"][0], "bounds must not overlap"

def allocate(category, mu, cov):
    cfg = CONFIG[category]; n = len(mu)
    bounds = [(MIN_W, cfg["max_w"])] * n
    bounds[EQ] = (max(MIN_W, cfg["eq"][0]), min(cfg["max_w"], cfg["eq"][1]))
    obj = lambda w: -(mu @ w - 0.5 * cfg["lam"] * w @ cov @ w)
    jac = lambda w: -(mu - cfg["lam"] * cov @ w)
    w0 = np.array([np.mean(b) for b in bounds]); w0 /= w0.sum()
    res = minimize(obj, w0, jac=jac, bounds=bounds, method="SLSQP",
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x, 0, None); w /= w.sum()
    return w, res.success

def split_pct(w, step=5):
    # round to multiples of `step` % (kept summing to 100) for a clean display in the app
    r = np.round(w * 100 / step) * step
    r[np.argmax(r)] += 100 - r.sum()
    return r.astype(int)

## 3. Point-estimate splits

In [ ]:
rows = {}
for cat in CONFIG:
    w, ok = allocate(cat, mu, cov)
    if not ok: print("WARNING: solver did not converge for", cat)
    rows[cat] = dict(zip(ASSETS, split_pct(w)))
    rows[cat]["exp. return %"] = round(float(mu @ w) * 100, 2)
    rows[cat]["volatility %"] = round(float(np.sqrt(w @ cov @ w)) * 100, 2)
point = pd.DataFrame(rows).T
point

In [ ]:
# sanity checks the report can cite
eqs = {c: allocate(c, mu, cov)[0][EQ] for c in CONFIG}
assert eqs["Conservative"] <= eqs["Moderate"] <= eqs["Aggressive"], eqs
vols = {c: np.sqrt(allocate(c, mu, cov)[0] @ cov @ allocate(c, mu, cov)[0]) for c in CONFIG}
print("Equity weight by category:", {k: round(v, 3) for k, v in eqs.items()})
print("Portfolio volatility by category:", {k: round(v, 4) for k, v in vols.items()})
if not (vols["Conservative"] <= vols["Moderate"] <= vols["Aggressive"]):
    print("NOTE: volatility is not ordered. Check the data, since the bounds only guarantee ordered equity weight.")
if eqs["Aggressive"] <= CONFIG["Aggressive"]["eq"][0] + 1e-6:
    print("NOTE: Aggressive sits on its equity floor, so the optimiser prefers less equity than the floor allows. "
          "Likely cause: NASI ignores dividends (see Step 2). Report this as a finding or add a sourced dividend yield.")

## 4. Stability test (200 bootstrap versions)
If mu and cov change slightly, does the split change a lot? A recommendation that jumps around on noise is not trustworthy. We run the allocator on each of the 200 bootstrap versions from Step 2.

In [ ]:
W = {c: [] for c in CONFIG}; fails = 0
for b in range(len(boot_mu)):
    for c in CONFIG:
        w, ok = allocate(c, boot_mu[b], boot_cov[b]); fails += (not ok); W[c].append(w)
W = {c: np.array(v) for c, v in W.items()}
print("Solver failures:", fails, "of", len(boot_mu) * len(CONFIG))

rows = []
for c in CONFIG:
    for i, a in enumerate(ASSETS):
        x = W[c][:, i] * 100
        rows.append([c, a, x.mean(), x.std(), np.percentile(x, 5), np.percentile(x, 95)])
stab = pd.DataFrame(rows, columns=["category", "asset", "mean %", "std %", "5th pct %", "95th pct %"]).round(1)
stab

In [ ]:
# one headline number per category: average total change in the split vs the point estimate
for c in CONFIG:
    base = allocate(c, mu, cov)[0]
    shift = np.abs(W[c] - base).sum(axis=1) / 2 * 100     # % of the portfolio that moves
    print(f"{c:13s} median shift {np.median(shift):5.1f}% | 95th pct {np.percentile(shift, 95):5.1f}% of the portfolio")

# ordering must hold in EVERY bootstrap sample
ok_order = np.mean((W["Conservative"][:, EQ] <= W["Moderate"][:, EQ] + 1e-9) &
                   (W["Moderate"][:, EQ] <= W["Aggressive"][:, EQ] + 1e-9))
print(f"Conservative <= Moderate <= Aggressive in equity weight: {ok_order*100:.0f}% of samples")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, c in zip(axes, CONFIG):
    ax.boxplot([W[c][:, i] * 100 for i in range(len(ASSETS))], showfliers=False)
    ax.set_xticks(range(1, len(ASSETS) + 1)); ax.set_xticklabels(ASSETS)
    ax.set_title(c); ax.set_ylim(0, 100)
axes[0].set_ylabel("weight % across 200 resamples")
plt.tight_layout(); plt.show()

## 5. Save for the app
`allocation_table.json` holds the point-estimate splits (the app can ship these as a lookup, so no server is needed). `allocation_stability.csv` is for the report.

In [ ]:
out = {"assets": ASSETS, "standin_data": USING_STANDIN, "config": CONFIG,
       "splits_pct": {c: dict(zip(ASSETS, map(int, split_pct(allocate(c, mu, cov)[0])))) for c in CONFIG},
       "disclaimer": "Educational use only. Not financial advice."}
json.dump(out, open("allocation_table.json", "w"), indent=2)
stab.to_csv("allocation_stability.csv", index=False)
print(json.dumps(out["splits_pct"], indent=2))
if USING_STANDIN: print("\n*** These came from STAND-IN data. Re-run after uploading the real asset_stats.json. ***")
try:
    from google.colab import files
    files.download("allocation_table.json"); files.download("allocation_stability.csv")
except ImportError:
    pass

## What to write in your report
- **Method:** mean-variance optimisation (Markowitz). Closed-form objective, solved numerically with bounds. Not a trained model.
- **Personalisation:** the risk category sets the risk aversion λ and the weight bounds. Conservative's equity cap sits below Aggressive's floor, so the ordering holds by design.
- **Stability:** re-ran the allocator on 200 block-bootstrap versions of mu and cov. Quote the median and 95th percentile shift.
- **Limitations:** bounds and λ are chosen by the authors, not learned. Short data history. Inputs inherit Step 2's issues (price-only NASI, missing NSE years, MMF series averaged across fund houses). With two assets the split is effectively a single number, the equity share. Educational use only, not financial advice.